In [1]:
# ============================
# W15e – Task 2: Isotherms & ps
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Load data and clean up
# ----------------------------

# Adjust the path if needed
data_file = "Data.csv"

df = pd.read_csv(data_file)

# Strip whitespace from column names (there is a trailing space after 'C' in the CSV)
df.columns = [c.strip() for c in df.columns]

# Rename for convenience
df = df.rename(columns={
    "Set Temperature (C)": "T_set_C",
    "Measured Temperature": "T_meas_C",
    "Volume (mm)": "V_mm",
    "Pressure (bar)": "p_bar"
})

# Add instrumental uncertainties
u_p_bar = 0.25   # pressure uncertainty in bar
u_V_mm  = 0.05   # volume uncertainty in mm

df["u_p_bar"] = u_p_bar
df["u_V_mm"]  = u_V_mm

# Check unique set temperatures (these define the isotherms)
temps = np.sort(df["T_set_C"].unique())
print("Isotherms at set temperatures (°C):", temps)


# ------------------------------------------------------------
# 2. Helper: find saturation pressure (Maxwell plateau region)
# ------------------------------------------------------------

def find_plateau_ps(group, window=3):
    """
    Find the flattest 'window' consecutive points in the isotherm
    and define the saturation pressure ps as the mean pressure
    in that window.

    Returns:
        ps_mean      : mean pressure in bar
        u_ps_bar     : combined uncertainty in ps (instrument + statistics)
        idx_window   : index positions within the group corresponding to plateau
    """
    # Sort by volume (largest -> smallest) for consistency
    g = group.sort_values("V_mm", ascending=False).reset_index(drop=True)
    p = g["p_bar"].to_numpy()

    if len(p) < window:
        raise ValueError("Not enough points in this isotherm for the chosen window size.")

    best_idx = None
    best_rel_std = np.inf

    # Slide a window of length 'window' across the isotherm
    for i in range(len(p) - window + 1):
        w = p[i:i+window]
        # relative standard deviation of the window
        rel_std = np.std(w, ddof=1) / np.mean(w)
        if rel_std < best_rel_std:
            best_rel_std = rel_std
            best_idx = i

    # Indices of the chosen plateau window
    idx_window = list(range(best_idx, best_idx + window))

    # Plateau pressures
    p_window = p[idx_window]
    ps_mean = np.mean(p_window)

    # Statistical uncertainty of the mean in the window
    if window > 1:
        u_stat = np.std(p_window, ddof=1) / np.sqrt(window)
    else:
        u_stat = 0.0

    # Combine instrumental uncertainty (u_p_bar) and statistical one in quadrature
    u_ps_bar = np.sqrt(u_p_bar**2 + u_stat**2)

    # Map plateau indices back to the original DataFrame indices
    original_indices = g.index[idx_window].to_list()

    return ps_mean, u_ps_bar, original_indices


# ---------------------------------------------------------
# 3. Loop over isotherms: plot p(V) and determine ps(T)
# ---------------------------------------------------------

ps_results = []  # will become a table of saturation pressures

for T in temps:
    iso = df[df["T_set_C"] == T].copy()
    iso = iso.sort_values("V_mm", ascending=False)  # large -> small volume

    # Find plateau and saturation pressure
    ps_mean, u_ps_bar, plateau_idx = find_plateau_ps(iso, window=3)
    ps_results.append({
        "T_set_C": T,
        "ps_bar": ps_mean,
        "u_ps_bar": u_ps_bar
    })

    # ---------------------- Plot for this isotherm ----------------------
    fig, ax = plt.subplots(figsize=(5, 4))

    # Error bars: black symbols, red error bars
    ax.errorbar(
        iso["V_mm"], iso["p_bar"],
        xerr=iso["u_V_mm"], yerr=iso["u_p_bar"],
        fmt="o", markersize=4,
        color="black", ecolor="red", capsize=2, linewidth=1
    )

    # Draw horizontal red line at ps (Maxwell plateau)
    V_min, V_max = iso["V_mm"].min(), iso["V_mm"].max()
    ax.hlines(ps_mean, V_min, V_max, colors="red", linestyles="--")

    # Axis labels only (no title)
    ax.set_xlabel("Volume / mm")
    ax.set_ylabel("Pressure / bar")

    # Optional: small grid
    ax.grid(True, linestyle=":", linewidth=0.5)

    # Save figure
    outname = f"isotherm_{int(T)}C.png"
    plt.tight_layout()
    plt.savefig(outname, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved {outname}  (ps = {ps_mean:.2f} ± {u_ps_bar:.2f} bar)")


# -------------------------------------------------------
# 4. Combined overview plot with all isotherms + ps lines
# -------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))

for T in temps:
    iso = df[df["T_set_C"] == T].copy()
    iso = iso.sort_values("V_mm", ascending=False)

    # Plot: black line & markers, no title
    ax.plot(
        iso["V_mm"], iso["p_bar"],
        "o-", markersize=3,
        color="black", linewidth=1, alpha=0.7
    )

# Draw one red horizontal line per isotherm at ps
for res in ps_results:
    T = res["T_set_C"]
    ps = res["ps_bar"]
    iso = df[df["T_set_C"] == T]
    V_min, V_max = iso["V_mm"].min(), iso["V_mm"].max()
    ax.hlines(ps, V_min, V_max, colors="red", linestyles="--", linewidth=0.6)

ax.set_xlabel("Volume / mm")
ax.set_ylabel("Pressure / bar")
ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.savefig("isotherms_all.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Saved isotherms_all.png")


# ----------------------------------------------
# 5. Create a table of ps(T) with uncertainties
# ----------------------------------------------

ps_df = pd.DataFrame(ps_results).sort_values("T_set_C")
print("\nSaturation vapour pressures (from plateau analysis):")
print(ps_df.to_string(index=False))

# If you want to save this table for later tasks:
ps_df.to_csv("ps_results_task2.csv", index=False)

Isotherms at set temperatures (°C): [20 33 36 39 42 45 47 50]
Saved isotherm_20C.png  (ps = 15.45 ± 0.27 bar)
Saved isotherm_33C.png  (ps = 28.17 ± 0.30 bar)
Saved isotherm_36C.png  (ps = 30.00 ± 0.25 bar)
Saved isotherm_39C.png  (ps = 32.00 ± 0.25 bar)
Saved isotherm_42C.png  (ps = 34.67 ± 0.30 bar)
Saved isotherm_45C.png  (ps = 37.00 ± 0.25 bar)
Saved isotherm_47C.png  (ps = 38.67 ± 0.51 bar)
Saved isotherm_50C.png  (ps = 40.58 ± 0.44 bar)
Saved isotherms_all.png

Saturation vapour pressures (from plateau analysis):
 T_set_C    ps_bar  u_ps_bar
      20 15.450000  0.270801
      33 28.166667  0.300463
      36 30.000000  0.250000
      39 32.000000  0.250000
      42 34.666667  0.300463
      45 37.000000  0.250000
      47 38.666667  0.506897
      50 40.583333  0.440959


In [2]:
# ============================
# W15e – Task 3: ln(ps) vs 1/T
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

R = 8.314462618  # J/(mol K)

# -------------------------------------------------
# 1. Load saturation pressures and original data
# -------------------------------------------------

ps_file   = "ps_results_task2.csv"  # created in Task 2
data_file = "Data.csv"

ps_df = pd.read_csv(ps_file)
data  = pd.read_csv(data_file)

# Clean column names in the raw data, as before
data.columns = [c.strip() for c in data.columns]
data = data.rename(columns={
    "Set Temperature (C)": "T_set_C",
    "Measured Temperature": "T_meas_C",
    "Volume (mm)": "V_mm",
    "Pressure (bar)": "p_bar"
})

# -------------------------------------------------
# 2. Use MEASURED temperatures: mean per isotherm
# -------------------------------------------------

T_meas_mean = (
    data
    .groupby("T_set_C")["T_meas_C"]
    .mean()
    .reset_index()
    .rename(columns={"T_meas_C": "T_meas_mean_C"})
)

# Merge with ps results
ps_df = ps_df.merge(T_meas_mean, on="T_set_C", how="inner")

# Convert to Kelvin and compute 1/T
ps_df["T_meas_K"] = ps_df["T_meas_mean_C"] + 273.15
ps_df["invT"]     = 1.0 / ps_df["T_meas_K"]

# -------------------------------------------------
# 3. Compute ln(ps) and its uncertainty
# -------------------------------------------------
# We can keep pressure in bar: the choice of units only shifts the intercept,
# not the slope, and therefore not Q.

ps_df["ln_ps"] = np.log(ps_df["ps_bar"])

# Uncertainty propagation: u(ln p) = u(p)/p
ps_df["u_ln_ps"] = ps_df["u_ps_bar"] / ps_df["ps_bar"]

print("Data for ln(ps) vs 1/T (measured temperatures):")
print(ps_df[["T_set_C", "T_meas_mean_C", "ps_bar", "u_ps_bar", "ln_ps", "u_ln_ps"]]
      .to_string(index=False))


# -------------------------------------------------
# 4. Weighted linear regression: ln(ps) = a + b * (1/T)
# -------------------------------------------------

x = ps_df["invT"].to_numpy()
y = ps_df["ln_ps"].to_numpy()
sigma_y = ps_df["u_ln_ps"].to_numpy()

# Build design matrix for y = a + b x
X = np.vstack((np.ones_like(x), x)).T

# Weight matrix W = diag(1/sigma^2)
w = 1.0 / (sigma_y**2)
W = np.diag(w)

# Normal equations: beta = (X^T W X)^(-1) X^T W y
XTWX = X.T @ W @ X
XTWy = X.T @ W @ y

cov_beta = np.linalg.inv(XTWX)
beta = cov_beta @ XTWy

a, b = beta  # intercept, slope
u_a, u_b = np.sqrt(np.diag(cov_beta))

# Goodness of fit: chi^2 and reduced chi^2
y_fit = a + b * x
chi2 = np.sum(((y - y_fit)/sigma_y)**2)
dof = len(x) - 2
chi2_red = chi2 / dof

print("\nWeighted linear regression ln(ps) = a + b * (1/T):")
print(f"a = {a:.5f} ± {u_a:.5f}")
print(f"b = {b:.5f} ± {u_b:.5f}   (units: K)")
print(f"chi^2 = {chi2:.2f},  dof = {dof},  chi^2_red = {chi2_red:.2f}")

# -------------------------------------------------
# 5. Latent heat from slope
#    ln(ps) = ln(ps0) - Q/(R T)  =>  b = -Q/R
# -------------------------------------------------

Q = -R * b       # J/mol
u_Q = R * u_b    # J/mol

print("\nMolar latent heat from slope:")
print(f"Q = {Q/1000:.2f} ± {u_Q/1000:.2f} kJ/mol")


# -------------------------------------------------
# 6. Plot ln(ps) vs 1/T with fit line, save PNG
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

# Data with error bars
ax.errorbar(
    ps_df["invT"], ps_df["ln_ps"],
    yerr=ps_df["u_ln_ps"],
    fmt="o", markersize=4, capsize=2
)

# Fit line over the range of x
x_fit = np.linspace(ps_df["invT"].min()*0.98, ps_df["invT"].max()*1.02, 200)
y_fit_line = a + b * x_fit
ax.plot(x_fit, y_fit_line)

ax.set_xlabel(r"$1/T\;(\mathrm{K}^{-1})$")
ax.set_ylabel(r"$\ln p_s$")

ax.grid(True, linestyle=":", linewidth=0.5)
plt.tight_layout()
plt.savefig("lnps_vs_invT_task3.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved lnps_vs_invT_task3.png")

Data for ln(ps) vs 1/T (measured temperatures):
 T_set_C  T_meas_mean_C    ps_bar  u_ps_bar    ln_ps  u_ln_ps
      20      21.270588 15.450000  0.270801 2.737609 0.017528
      33      32.985714 28.166667  0.300463 3.338139 0.010667
      36      36.011111 30.000000  0.250000 3.401197 0.008333
      39      39.118750 32.000000  0.250000 3.465736 0.007812
      42      42.107692 34.666667  0.300463 3.545779 0.008667
      45      45.069231 37.000000  0.250000 3.610918 0.006757
      47      47.058333 38.666667  0.506897 3.654978 0.013109
      50      50.054545 40.583333  0.440959 3.703357 0.010866

Weighted linear regression ln(ps) = a + b * (1/T):
a = 12.29530 ± 0.16387
b = -2760.66938 ± 51.40949   (units: K)
chi^2 = 185.22,  dof = 6,  chi^2_red = 30.87

Molar latent heat from slope:
Q = 22.95 ± 0.43 kJ/mol
Saved lnps_vs_invT_task3.png


In [3]:
# ============================
# W15e – Task 4: Amount of substance n
# ============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

R = 8.314462618  # J/(mol K)

# ------------------------------------
# 1. Load data and clean column names
# ------------------------------------

data_file = "Data.csv"
df = pd.read_csv(data_file)

df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={
    "Set Temperature (C)": "T_set_C",
    "Measured Temperature": "T_meas_C",
    "Volume (mm)": "V_unit",
    "Pressure (bar)": "p_bar"
})

# Instrumental uncertainties (same as before)
u_p_bar = 0.25      # bar
u_V_unit = 0.05     # "mm" – treated as volume units

# ------------------------------------
# 2. Select highest-temperature isotherm (50 °C)
# ------------------------------------

T_high = 50  # set temperature for highest isotherm
iso = df[df["T_set_C"] == T_high].copy()

# Sort by volume: largest -> smallest (ideal gas at large V)
iso = iso.sort_values("V_unit", ascending=False).reset_index(drop=True)

print(f"Number of points in {T_high} °C isotherm:", len(iso))

# Mean measured temperature for this isotherm (in Kelvin)
T_meas_mean_C = iso["T_meas_C"].mean()
T_meas_K = T_meas_mean_C + 273.15
print(f"Mean measured temperature for {T_high} °C isotherm: "
      f"{T_meas_mean_C:.3f} °C = {T_meas_K:.3f} K")

# ------------------------------------
# 3. Convert to SI units: p in Pa, V in m^3
# ------------------------------------
# Assumption: one "V_unit" (labelled mm on scale) corresponds to 1 cm^3
# => 1 cm^3 = 1e-6 m^3. Adjust this factor if your calibration is different.

V_cm3 = iso["V_unit"].to_numpy()          # treat numerical values as cm^3
V = V_cm3 * 1e-6                          # m^3
p = iso["p_bar"].to_numpy() * 1e5         # Pa

# Uncertainties in SI
u_p = u_p_bar * 1e5          # Pa
u_V = u_V_unit * 1e-6        # m^3

# ------------------------------------
# 4. Choose "gas-like" region for fit
# ------------------------------------
# Use the N largest-volume points (lowest pressures).
# You can change N if you like; N=10 is a reasonable default.

N = min(10, len(iso))  # in case there are fewer points
V_fit = V[:N]
p_fit = p[:N]

# ------------------------------------
# 5. Compute y = pV and x = 1/V with uncertainties
# ------------------------------------

x = 1.0 / V_fit              # 1/m^3
y = p_fit * V_fit            # J (since Pa * m^3)

# Uncertainty of y = pV  (assuming p and V uncorrelated)
# u_y^2 = (V u_p)^2 + (p u_V)^2
u_y = np.sqrt((V_fit * u_p)**2 + (p_fit * u_V)**2)

# (Optional) uncertainty of x = 1/V for information
u_x = u_V / (V_fit**2)

# ------------------------------------
# 6. Weighted linear regression: y = a + b x
# ------------------------------------

# Design matrix
X = np.vstack((np.ones_like(x), x)).T

# Weights = 1/u_y^2
w = 1.0 / (u_y**2)
W = np.diag(w)

XTWX = X.T @ W @ X
XTWy = X.T @ W @ y

cov_beta = np.linalg.inv(XTWX)
beta = cov_beta @ XTWy

a, b = beta           # intercept (a), slope (b)
u_a, u_b = np.sqrt(np.diag(cov_beta))

# Goodness of fit
y_fit = a + b * x
chi2 = np.sum(((y - y_fit) / u_y)**2)
dof = len(x) - 2
chi2_red = chi2 / dof

print("\nWeighted linear regression  y = a + b (1/V):")
print(f"a = (pV)_V->∞ = {a:.4e} ± {u_a:.4e} J")
print(f"b = {b:.4e} ± {u_b:.4e} J·m^3")
print(f"chi^2 = {chi2:.2f},  dof = {dof},  chi^2_red = {chi2_red:.2f}")

# ------------------------------------
# 7. Amount of substance n and uncertainty
# ------------------------------------
# From ideal-gas limit:  a = n R T  =>  n = a / (R T)

n = a / (R * T_meas_K)           # mol
u_n = u_a / (R * T_meas_K)       # mol (temperature uncertainty neglected)

print("\nAmount of substance n from intercept:")
print(f"n = {n:.4e} ± {u_n:.4e} mol")

# ------------------------------------
# 8. Plot pV vs 1/V with fit line and save PNG
# ------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

# Data with error bars
ax.errorbar(
    x, y,
    yerr=u_y,
    fmt="o", markersize=4, capsize=2,
    label="data"
)

# Fit line on full x-range
x_line = np.linspace(x.min()*0.95, x.max()*1.05, 200)
y_line = a + b * x_line
ax.plot(x_line, y_line, label="fit")

ax.set_xlabel(r"$1/V\;(\mathrm{m}^{-3})$")
ax.set_ylabel(r"$pV\;(\mathrm{J})$")
ax.grid(True, linestyle=":", linewidth=0.5)
ax.legend()

plt.tight_layout()
plt.savefig("pV_vs_invV_task4.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved pV_vs_invV_task4.png")

Number of points in 50 °C isotherm: 11
Mean measured temperature for 50 °C isotherm: 50.055 °C = 323.205 K

Weighted linear regression  y = a + b (1/V):
a = (pV)_V->∞ = 3.5710e+00 ± 6.6873e-02 J
b = -7.5988e-07 ± 6.3536e-08 J·m^3
chi^2 = 5.34,  dof = 8,  chi^2_red = 0.67

Amount of substance n from intercept:
n = 1.3289e-03 ± 2.4885e-05 mol
Saved pV_vs_invV_task4.png


In [4]:
# ==========================================
# W15e – Task 5: Temperature-dependent Q23(T)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

R = 8.314462618  # J/(mol K)

# ------------------------------------------
# 1. Load data (ps from Task 2 and raw data)
# ------------------------------------------

ps_df = pd.read_csv("ps_results_task2.csv")
data  = pd.read_csv("Data.csv")

# Clean column names and rename for convenience
data.columns = [c.strip() for c in data.columns]
data = data.rename(columns={
    "Set Temperature (C)": "T_set_C",
    "Measured Temperature": "T_meas_C",
    "Volume (mm)": "V_unit",      # capillary scale
    "Pressure (bar)": "p_bar"
})

# Instrumental uncertainties
u_p_bar  = 0.25       # bar
u_V_unit = 0.05       # "mm" ~ cm^3

# ------------------------------------------
# 2. Add measured temperatures to ps table
# ------------------------------------------

T_meas_mean = (
    data.groupby("T_set_C")["T_meas_C"]
    .mean()
    .reset_index()
    .rename(columns={"T_meas_C": "T_meas_mean_C"})
)

ps_df = ps_df.merge(T_meas_mean, on="T_set_C", how="inner")
ps_df["T_meas_K"] = ps_df["T_meas_mean_C"] + 273.15

# ------------------------------------------
# 3. Re-identify plateau window to get V_G, V_F
# ------------------------------------------

def find_plateau_indices(group, window=3):
    """
    Reproduce the Task-2 plateau finder.
    Returns the original indices of the flattest 'window' in p_bar.
    """
    g = group.sort_values("V_unit", ascending=False).reset_index()
    p = g["p_bar"].to_numpy()
    if len(p) < window:
        raise ValueError("Too few data points in isotherm.")
    best_i = None
    best_rel = np.inf
    for i in range(len(p) - window + 1):
        w = p[i:i+window]
        rel = np.std(w, ddof=1) / np.mean(w)
        if rel < best_rel:
            best_rel = rel
            best_i = i
    idx_window = list(range(best_i, best_i + window))
    return g.loc[idx_window, "index"].to_list()

VG_list = []
VF_list = []

for Tset in ps_df["T_set_C"]:
    grp = data[data["T_set_C"] == Tset]
    idxs = find_plateau_indices(grp, window=3)
    plateau = grp.loc[idxs].sort_values("V_unit", ascending=False)
    V_G = plateau["V_unit"].iloc[0]   # largest V in plateau -> gas
    V_F = plateau["V_unit"].iloc[-1]  # smallest V in plateau -> liquid
    VG_list.append(V_G)
    VF_list.append(V_F)

ps_df["V_G_unit"] = VG_list
ps_df["V_F_unit"] = VF_list

# ------------------------------------------
# 4. Amount of substance n (from Task 4)
# ------------------------------------------

n   = 1.3289e-3   # mol
u_n = 2.4885e-5   # mol

# Convert volumes to SI: units are treated as cm^3
ps_df["V_G"] = ps_df["V_G_unit"] * 1e-6   # m^3
ps_df["V_F"] = ps_df["V_F_unit"] * 1e-6   # m^3

u_V = u_V_unit * 1e-6                     # m^3

# ------------------------------------------
# 5. Molar volumes Vm_G, Vm_F and ΔVm
# ------------------------------------------

Vm_G = ps_df["V_G"].to_numpy() / n
Vm_F = ps_df["V_F"].to_numpy() / n

# Uncertainties in molar volumes:
# Vm = V/n -> u^2(Vm) = (1/n)^2 u_V^2 + (V/n^2)^2 u_n^2
u_Vm_G = np.sqrt((u_V / n)**2 + (ps_df["V_G"].to_numpy() * u_n / n**2)**2)
u_Vm_F = np.sqrt((u_V / n)**2 + (ps_df["V_F"].to_numpy() * u_n / n**2)**2)

ps_df["Vm_G"]   = Vm_G
ps_df["Vm_F"]   = Vm_F
ps_df["u_Vm_G"] = u_Vm_G
ps_df["u_Vm_F"] = u_Vm_F

ps_df["dVm"]    = ps_df["Vm_G"] - ps_df["Vm_F"]
ps_df["u_dVm"]  = np.sqrt(ps_df["u_Vm_G"]**2 + ps_df["u_Vm_F"]**2)

# ------------------------------------------
# 6. Refit ln(ps) vs 1/T to get Q_eff, u_Q_eff
# ------------------------------------------

T   = ps_df["T_meas_K"].to_numpy()
invT = 1.0 / T
ps   = ps_df["ps_bar"].to_numpy()
u_ps = ps_df["u_ps_bar"].to_numpy()

ln_ps   = np.log(ps)
u_ln_ps = u_ps / ps

X = np.vstack((np.ones_like(invT), invT)).T
w = 1.0 / (u_ln_ps**2)
W = np.diag(w)

XTWX  = X.T @ W @ X
XTWy  = X.T @ W @ ln_ps
cov_beta = np.linalg.inv(XTWX)
beta     = cov_beta @ XTWy
a_ln, b_ln = beta          # ln(ps) = a_ln + b_ln / T
u_a_ln, u_b_ln = np.sqrt(np.diag(cov_beta))

Q_eff   = -R * b_ln        # J/mol
u_Q_eff =  R * u_b_ln

print(f"Effective latent heat from fit: Q_eff = {Q_eff/1000:.2f} ± {u_Q_eff/1000:.2f} kJ/mol")

# ------------------------------------------
# 7. Compute dp_s/dT and uncertainty
#     From ps(T) = ps0 exp(-Q_eff/(RT))
#     => dp/dT = ps * Q_eff / (R T^2)
# ------------------------------------------

dpdT = ps * Q_eff / (R * T**2)

u_dpdT = np.sqrt(
    (Q_eff / (R * T**2))**2 * u_ps**2 +
    (ps / (R * T**2))**2 * u_Q_eff**2
)

ps_df["dpdT"]    = dpdT
ps_df["u_dpdT"]  = u_dpdT

# ------------------------------------------
# 8. Q(T) = T ΔVm dp_s/dT with uncertainty
# ------------------------------------------

u_T = 0.1  # K (approx. thermometer / stability)

dVm   = ps_df["dVm"].to_numpy()
u_dVm = ps_df["u_dVm"].to_numpy()

Q_T   = T * dVm * dpdT                 # J/mol
u_Q_T = np.sqrt(
    (dVm * dpdT * u_T)**2 +           # from T
    (T * dpdT * u_dVm)**2 +           # from ΔVm
    (T * dVm * u_dpdT)**2             # from dp/dT
)

ps_df["Q_T"]   = Q_T
ps_df["u_Q_T"] = u_Q_T

# also store in kJ/mol for nicer numbers
ps_df["Q_T_kJ"]   = ps_df["Q_T"] / 1000.0
ps_df["u_Q_T_kJ"] = ps_df["u_Q_T"] / 1000.0

print("\nTemperature-dependent latent heat Q(T):")
print(ps_df[["T_set_C", "T_meas_mean_C", "Q_T_kJ", "u_Q_T_kJ"]].to_string(index=False))

# ------------------------------------------
# 9. Reduced temperature T/Tk (van der Waals)
# ------------------------------------------

a_vdw = 0.79        # N m^4 / mol^2
b_vdw = 0.88e-4     # m^3 / mol
T_k   = 8 * a_vdw / (27 * b_vdw * R)

T_red = T / T_k
ps_df["T_red"] = T_red

print(f"\nCritical temperature from van der Waals: T_k = {T_k:.2f} K")
print("Reduced temperatures T/T_k:", T_red)

# ------------------------------------------
# 10. Power-law fit: Q(T) = Q0 [1 - T/Tk]^beta
#      -> ln Q = ln Q0 + beta * ln(1 - T/Tk)
# ------------------------------------------

x_pw = np.log(1.0 - T_red)
y_pw = np.log(ps_df["Q_T"].to_numpy())        # natural log of Q in J/mol
sigma_y_pw = ps_df["u_Q_T"].to_numpy() / ps_df["Q_T"].to_numpy()

Xp = np.vstack((np.ones_like(x_pw), x_pw)).T
w_pw = 1.0 / (sigma_y_pw**2)
Wp   = np.diag(w_pw)

XTWX_pw = Xp.T @ Wp @ Xp
XTWy_pw = Xp.T @ Wp @ y_pw
cov_pw  = np.linalg.inv(XTWX_pw)
beta_pw = cov_pw @ XTWy_pw
ln_Q0, beta_exp = beta_pw
u_ln_Q0, u_beta = np.sqrt(np.diag(cov_pw))

Q0 = np.exp(ln_Q0)
u_Q0 = Q0 * u_ln_Q0

print("\nPower-law fit Q(T) = Q0 [1 - T/Tk]^beta:")
print(f"Q0  = {Q0/1000:.2f} ± {u_Q0/1000:.2f} kJ/mol")
print(f"beta = {beta_exp:.3f} ± {u_beta:.3f}")

# ------------------------------------------
# 11. Plot Q(T) vs T/Tk with power-law fit
# ------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

# Data: black points with red error bars
ax.errorbar(
    ps_df["T_red"], ps_df["Q_T_kJ"],
    yerr=ps_df["u_Q_T_kJ"],
    fmt="o", markersize=4,
    color="black", ecolor="red", capsize=2, linewidth=1
)

# Fit curve (red line)
Tred_fit = np.linspace(ps_df["T_red"].min()*0.98,
                       ps_df["T_red"].max()*1.02, 300)
Q_fit_kJ = (Q0 * (1.0 - Tred_fit)**beta_exp) / 1000.0
ax.plot(Tred_fit, Q_fit_kJ, color="red", linewidth=1)

ax.set_xlabel(r"$T/T_k$")
ax.set_ylabel(r"$Q_{23}(T)\;(\mathrm{kJ\,mol^{-1}})$")
ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.savefig("Q_vs_Tred_task5.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved Q_vs_Tred_task5.png")

# ------------------------------------------
# 12. Log–log plot: ln Q vs ln(1 - T/Tk)
# ------------------------------------------

fig, ax = plt.subplots(figsize=(5, 4))

ax.errorbar(
    x_pw, y_pw,
    yerr=sigma_y_pw,
    fmt="o", markersize=4,
    color="black", ecolor="red", capsize=2, linewidth=1
)

x_fit_log = np.linspace(x_pw.min()*1.02, x_pw.max()*0.98, 300)
y_fit_log = ln_Q0 + beta_exp * x_fit_log
ax.plot(x_fit_log, y_fit_log, color="red", linewidth=1)

ax.set_xlabel(r"$\ln(1 - T/T_k)$")
ax.set_ylabel(r"$\ln Q_{23}(T)$")
ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()
plt.savefig("logQ_powerlaw_task5.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved logQ_powerlaw_task5.png")

# ------------------------------------------
# 13. Save table for later use (optional)
# ------------------------------------------

ps_df.to_csv("Q_T_task5_results.csv", index=False)

Effective latent heat from fit: Q_eff = 22.95 ± 0.43 kJ/mol

Temperature-dependent latent heat Q(T):
 T_set_C  T_meas_mean_C   Q_T_kJ  u_Q_T_kJ
      20      21.270588 0.000033  0.000009
      33      32.985714 0.000029  0.000014
      36      36.011111 0.000020  0.000015
      39      39.118750 0.000021  0.000015
      42      42.107692 0.000023  0.000016
      45      45.069231 0.000024  0.000017
      47      47.058333 0.000050  0.000018
      50      50.054545 0.000026  0.000019

Critical temperature from van der Waals: T_k = 319.92 K
Reduced temperatures T/T_k: [0.92030487 0.95692421 0.96638104 0.97609495 0.98543784 0.99469507
 1.00091264 1.01027825]

Power-law fit Q(T) = Q0 [1 - T/Tk]^beta:
Q0  = nan ± nan kJ/mol
beta = nan ± nan


C:\Users\laiba\AppData\Local\Temp\ipykernel_9572\928754293.py:206: RuntimeWarning: invalid value encountered in log
  x_pw = np.log(1.0 - T_red)


Saved Q_vs_Tred_task5.png
Saved logQ_powerlaw_task5.png
